# Day 23：FastAPI 后端开发(上)：路由设计、Pydantic v2 校验与 REST API 实战（Jupyter Notebook 实战手册）

> **所属阶段**：第 2 阶段 · 第二阶段：大模型核心原理、Prompt工程与全栈 Web API ｜ **第 4 周**：第4周：全栈 Web 聊天应用与 FastAPI 后端工程  
> **GitLab 工单**：`STARMIND-023` ｜ **开发分支**：`feat/day23-fastapi-rest-chat`  
> **配套 HTML5 课件**：`courseware/day23/index.html` ｜ **独立版本快照**：`versions/day23/`

---

## 🎙️ 老陈开场旁白

早啊各位！今天咱们正式进入 Python AI 后端领域的绝对王者——**FastAPI**！为什么现在国内大厂和 AI 创业公司 90% 的模型服务网关都选 FastAPI 而不是 Flask？就凭三大杀手锏：原生 `async/await` 高并发异步能力、基于 **Pydantic v2**（Rust 核心）的极速强类型入参出参校验、以及自动生成的 OpenAPI `/docs` Swagger 交互文档！今天上午咱们把路由、路径参数、查询参数和 Pydantic v2 `Field`/`field_validator` 吃透，下午把咱们的星辰智脑对话服务封装成规范的 RESTful API，并用 `fastapi.testclient.TestClient` 做到不启端口也能毫秒级跑完所有 200/404/422 自动化测试！

### 承上启下
- **昨日基线**：昨天（Day 22）我们用一天时间速成了企业级聊天前端，产出了 `index.html`、`styles.css`、`chat_app.js` 并通过了 DOM 与防 XSS 校验。但现在前端页面发出的 `/api/v1/chat` 请求还没有真实的后端接口来承接！
- **明日预告**：明天（Day 24）将是第二阶段的收官大戏——FastAPI 后端开发（下）+ 数据库：我们将接入 SSE 流式接口、CORS 跨域、SQLite + SQLAlchemy 对话历史持久化，彻底跑通「网页版星辰 ChatGPT 克隆」前后端联调！


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day23'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-023）

星辰智脑需要向前端网页和内部业务系统（OA、钉钉机器人、客服工作台）暴露统一、强类型、文档自动化的 RESTful HTTP API，替代此前的本地 Python 函数调用。

**今日迭代目标：**
- 使用 Pydantic v2 定义严格的请求/响应模型：`ChatRequest`、`ChatResponse`、`ModelInfo`、`SessionDetailResponse`，包含字段范围约束与自定义 `field_validator`。
- 使用 FastAPI 构建完整 REST API：`/health`、`/api/v1/models`（查询参数过滤）、`/api/v1/chat`（POST 对话）、`/api/v1/sessions/{session_id}`（路径参数查询与删除）。
- 使用 `fastapi.testclient.TestClient` 编写完整 pytest 自动化测试，覆盖正常 200、参数非法 422、资源不存在 404 以及多轮会话状态校验。

**验收标准：**
- 非法入参（空消息、`temperature > 2.0`、未支持的 `model`）自动返回 HTTP 422 与清晰错误字段定位。
- 查询不存在的 `session_id` 返回标准 HTTP 404 JSON 错误，而非 500 堆栈泄露。
- `/api/v1/models?provider=DeepSeek` 查询参数能精准筛选模型列表，所有 `TestClient` 用例 100% 通过。

---

## 📚 课堂核心技术笔记

### 1. 1. Pydantic v2 核心语法与 v1 迁移避坑

FastAPI 现代版本全面拥抱 Pydantic v2：
- 用 `@field_validator('message')` + `@classmethod` 替代旧版 `@validator`；
- 用 `model_config = ConfigDict(str_strip_whitespace=True)` 替代旧版 `class Config:`；
- 用 `model.model_dump()` 替代旧版 `model.dict()`，用 `model.model_dump_json()` 替代 `.json()`。

> **⚠️ 老陈避坑**：常见坑：在 `Field(default=[])` 中放可变默认值虽在 Pydantic 里会被拷贝，但最佳实践仍是使用 `Field(default_factory=list)`，与 Python `dataclass` 保持风格一致。

### 2. 2. 路径参数、查询参数与请求体的区分规则

FastAPI 自动根据函数签名推断参数来源：
- 出现在路由模板 `{session_id}` 中的参数 -> **路径参数（Path）**；
- 声明为 Pydantic `BaseModel` 的参数 -> **JSON 请求体（Body）**；
- 其余基础类型（如 `provider: str | None = None, limit: int = 10`）-> **查询参数（Query）**。

> **⚠️ 老陈避坑**：URL 路由顺序坑：如果同时定义了 `/sessions/latest`（固定路径）和 `/sessions/{session_id}`（动态路径），固定路径必须写在前面，否则 `'latest'` 会被当成 `session_id` 吃掉！

### 3. 3. `response_model` 的安全过滤价值

在路由装饰器上声明 `@app.post('/api/v1/chat', response_model=ChatResponse)` 不仅能自动生成 Swagger 响应文档，更重要的是能充当**数据脱敏过滤器**——即使内部函数多返回了敏感字段（如 `internal_api_key` 或 `db_password`），FastAPI 也会严格按 `response_model` 裁剪掉多余字段，绝不外泄！

> **⚠️ 老陈避坑**：如果 `response_model` 定义了必填字段而接口漏返回了，FastAPI 会立刻报 500 `ResponseValidationError`，帮你第一时间发现契约破坏。


## 💻 实操步骤 1：步骤一：使用 Pydantic v2 定义星辰智脑 REST API 强类型请求与响应模型

- **目标文件**：`versions/day23/src/day23_schemas.py`
- **老陈代码旁白**：咱们先写契约层 `src/day23_schemas.py`，全部采用 Pydantic v2 现代写法（`ConfigDict`、`Field`、`field_validator`），把入参范围检查、空格裁剪和模型白名单验证一次性做扎实。


In [ ]:
"""
StarMind Day 23 - 步骤一：基于 Pydantic v2 的企业级 API 请求与响应数据模型
采用 ConfigDict、Field 与 @field_validator 现代规范。
"""

from typing import List, Literal
from pydantic import BaseModel, ConfigDict, Field, field_validator


SUPPORTED_MODELS = ("deepseek-chat", "qwen-max", "glm-4-plus")


class HealthResponse(BaseModel):
    status: Literal["ok"] = "ok"
    service: str = "starmind-api"
    version: str = "0.23.0"


class ModelInfo(BaseModel):
    model_id: str
    provider: str
    context_window: int
    supports_tools: bool = True


class ChatRequest(BaseModel):
    """AI 对话请求体（严格强类型与业务规则校验）。"""

    model_config = ConfigDict(str_strip_whitespace=True)

    session_id: str = Field(..., min_length=2, max_length=64, description="会话唯一标识")
    message: str = Field(..., min_length=1, max_length=4000, description="用户输入文本")
    model: str = Field(default="deepseek-chat", description="目标大模型 ID")
    temperature: float = Field(default=0.3, ge=0.0, le=2.0, description="采样温度 [0.0, 2.0]")
    max_tokens: int = Field(default=512, ge=16, le=8192, description="最大生成长度")

    @field_validator("model")
    @classmethod
    def validate_model_id(cls, value: str) -> str:
        if value not in SUPPORTED_MODELS:
            raise ValueError(f"不支持的模型 '{value}'，可选列表: {list(SUPPORTED_MODELS)}")
        return value


class MessageRecord(BaseModel):
    role: Literal["user", "assistant"]
    content: str


class ChatResponse(BaseModel):
    """AI 对话标准响应体。"""

    session_id: str
    model: str
    reply: str
    turn_count: int
    estimated_tokens: int


class SessionDetailResponse(BaseModel):
    """会话详情与历史记录响应体。"""

    session_id: str
    message_count: int
    messages: List[MessageRecord]


if __name__ == "__main__":
    req = ChatRequest(session_id="s_001", message="  杭州出差住宿报销上限是多少？  ")
    print("校验通过的请求模型:", req.model_dump())


### 🔍 步骤 1 代码深度解析

1. `ConfigDict(str_strip_whitespace=True)` 自动剥离所有字符串字段首尾空白，如果用户只传了 `'   '` 会直接触发 `min_length=1` 校验拦截。
2. `@field_validator('model')` 把支持的模型白名单校验内聚在 Schema 层，路由函数无需再写重复的 `if model not in ...`。
3. `Literal['user', 'assistant']` 和 `ge/le` 边界约束会自动体现在 Swagger `/docs` 文档中。

**预期运行输出参考：**
```text
校验通过的请求模型: {'session_id': 's_001', 'message': '杭州出差住宿报销上限是多少？', 'model': 'deepseek-chat', 'temperature': 0.3, 'max_tokens': 512}
```


## 💻 实操步骤 2：步骤二：使用 FastAPI 封装星辰智脑 RESTful 对话微服务（工厂模式）

- **目标文件**：`versions/day23/src/day23_fastapi_app.py`
- **老陈代码旁白**：契约就绪，咱们写 `src/day23_fastapi_app.py`！通过 `create_app()` 工厂函数构建完整的 FastAPI 应用，涵盖 `/health`、`/api/v1/models`（查询参数）、`/api/v1/chat`（POST 对话与多轮记忆）以及 `/api/v1/sessions/{session_id}`（路径参数查删）。


In [ ]:
"""
StarMind Day 23 - 步骤二：FastAPI 企业级 RESTful AI 对话服务
演示路径参数、查询参数、Pydantic v2 校验、HTTPException 规范错误与工厂模式隔离。
"""

from typing import Dict, List
from fastapi import FastAPI, HTTPException, Query

from src.day23_schemas import (
    ChatRequest,
    ChatResponse,
    HealthResponse,
    MessageRecord,
    ModelInfo,
    SessionDetailResponse,
)


BUILTIN_MODELS: List[ModelInfo] = [
    ModelInfo(model_id="deepseek-chat", provider="DeepSeek", context_window=64000, supports_tools=True),
    ModelInfo(model_id="qwen-max", provider="Alibaba", context_window=32000, supports_tools=True),
    ModelInfo(model_id="glm-4-plus", provider="Zhipu", context_window=128000, supports_tools=True),
]


def _generate_enterprise_reply(message: str, history: List[MessageRecord], model: str) -> str:
    """离线确定性企业问答回复生成器（无外网依赖，便于联调与自动化测试）。"""
    if "差旅" in message or "住宿" in message:
        return f"[{model}] 根据星辰数科 HR-001 制度，北京/上海/深圳/杭州出差住宿上限为 600 元/晚，需提供电子专票。"
    if "vpn" in message.lower():
        return f"[{model}] 请在 OA 系统发起『IT权限申请』，主管审批后 30 分钟内自动开通 WireGuard VPN。"
    if "上一句" in message or "刚才" in message:
        prev_users = [m.content for m in history if m.role == "user"]
        if prev_users:
            return f"[{model}] 您上一轮提到的问题是：「{prev_users[-1]}」。"
    return f"[{model}] 星辰智脑已收到您的咨询「{message}」，当前已为您记录至企业会话上下文。"


def create_app() -> FastAPI:
    """应用工厂函数：每次调用返回状态完全隔离的 FastAPI 实例。"""
    app = FastAPI(
        title="星辰智脑 StarMind REST API",
        description="杭州星辰数科企业级 AI 对话后端服务（Day 23 版本）",
        version="0.23.0",
    )

    # 实例级内存会话存储（Day 24 将升级为 SQLite + SQLAlchemy 持久化）
    sessions: Dict[str, List[MessageRecord]] = {}

    @app.get("/health", response_model=HealthResponse, tags=["System"])
    def health_check() -> HealthResponse:
        return HealthResponse()

    @app.get("/api/v1/models", response_model=List[ModelInfo], tags=["Models"])
    def list_models(
        provider: str | None = Query(default=None, description="按厂商名称过滤（不区分大小写）"),
        min_context: int = Query(default=0, ge=0, description="按最小上下文窗口过滤"),
    ) -> List[ModelInfo]:
        results = BUILTIN_MODELS
        if provider:
            results = [m for m in results if m.provider.lower() == provider.lower()]
        if min_context > 0:
            results = [m for m in results if m.context_window >= min_context]
        return results

    @app.post("/api/v1/chat", response_model=ChatResponse, tags=["Chat"])
    def chat_endpoint(req: ChatRequest) -> ChatResponse:
        history = sessions.setdefault(req.session_id, [])
        reply_text = _generate_enterprise_reply(req.message, history, req.model)
        # 按 max_tokens 做安全输出截断保护
        reply_text = reply_text[: req.max_tokens]

        history.append(MessageRecord(role="user", content=req.message))
        history.append(MessageRecord(role="assistant", content=reply_text))

        est_tokens = max(1, (len(req.message) + len(reply_text)) // 2)
        return ChatResponse(
            session_id=req.session_id,
            model=req.model,
            reply=reply_text,
            turn_count=len(history) // 2,
            estimated_tokens=est_tokens,
        )

    @app.get("/api/v1/sessions/{session_id}", response_model=SessionDetailResponse, tags=["Sessions"])
    def get_session(session_id: str) -> SessionDetailResponse:
        if session_id not in sessions:
            raise HTTPException(status_code=404, detail=f"会话 '{session_id}' 不存在")
        msgs = sessions[session_id]
        return SessionDetailResponse(session_id=session_id, message_count=len(msgs), messages=msgs)

    @app.delete("/api/v1/sessions/{session_id}", tags=["Sessions"])
    def delete_session(session_id: str) -> Dict[str, str]:
        if session_id not in sessions:
            raise HTTPException(status_code=404, detail=f"会话 '{session_id}' 不存在")
        del sessions[session_id]
        return {"status": "deleted", "session_id": session_id}

    return app


app = create_app()


if __name__ == "__main__":
    from fastapi.testclient import TestClient

    client = TestClient(create_app())
    print("健康检查:", client.get("/health").json())
    print("模型过滤:", client.get("/api/v1/models?provider=DeepSeek").json())
    resp = client.post("/api/v1/chat", json={"session_id": "demo_01", "message": "杭州出差住宿报销标准？"})
    print("对话响应:", resp.json())


### 🔍 步骤 2 代码深度解析

1. `create_app()` 将内存 `sessions` 字典封装在闭包内，每次调用 `create_app()` 都是全新实例，彻底告别单元测试间状态互相污染的顽疾。
2. `/api/v1/models` 演示了带校验的 `Query` 查询参数；`/api/v1/sessions/{session_id}` 演示了 `Path` 路径参数与 `HTTPException(status_code=404)` 标准报错。
3. 直接在 `__main__` 中用 `TestClient` 做冒烟测试，无需启动 uvicorn 进程即可秒级验证整条链路。

**预期运行输出参考：**
```text
健康检查: {'status': 'ok', 'service': 'starmind-api', 'version': '0.23.0'}
模型过滤: [{'model_id': 'deepseek-chat', 'provider': 'DeepSeek', 'context_window': 64000, 'supports_tools': True}]
对话响应: {'session_id': 'demo_01', 'model': 'deepseek-chat', 'reply': '[deepseek-chat] 根据星辰数科 HR-001 制度...', 'turn_count': 1, ...}
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day23/tests/test_day23.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 23 自动化单元测试：使用 FastAPI TestClient 全面验证路由、Pydantic v2 校验与会话状态。"""

from fastapi.testclient import TestClient
from src.day23_fastapi_app import create_app


def test_health_and_models_query_params():
    client = TestClient(create_app())

    r_health = client.get("/health")
    assert r_health.status_code == 200
    assert r_health.json()["status"] == "ok"

    r_all = client.get("/api/v1/models")
    assert r_all.status_code == 200
    assert len(r_all.json()) == 3

    r_filtered = client.get("/api/v1/models", params={"provider": "DeepSeek", "min_context": 50000})
    assert r_filtered.status_code == 200
    assert [m["model_id"] for m in r_filtered.json()] == ["deepseek-chat"]


def test_chat_multi_turn_and_session_lifecycle():
    client = TestClient(create_app())

    r1 = client.post("/api/v1/chat", json={"session_id": "s_100", "message": "杭州出差住宿怎么报销？"})
    assert r1.status_code == 200
    body1 = r1.json()
    assert body1["turn_count"] == 1
    assert "600 元/晚" in body1["reply"]

    r2 = client.post("/api/v1/chat", json={"session_id": "s_100", "message": "我刚才上一句问了什么？"})
    assert r2.status_code == 200
    assert "杭州出差住宿怎么报销" in r2.json()["reply"]
    assert r2.json()["turn_count"] == 2

    # 路径参数查询会话
    r_sess = client.get("/api/v1/sessions/s_100")
    assert r_sess.status_code == 200
    assert r_sess.json()["message_count"] == 4

    # 删除会话后再查询应返回 404
    r_del = client.delete("/api/v1/sessions/s_100")
    assert r_del.status_code == 200
    assert client.get("/api/v1/sessions/s_100").status_code == 404


def test_pydantic_v2_validation_errors_return_422():
    client = TestClient(create_app())

    # 空白消息触发 422
    assert client.post("/api/v1/chat", json={"session_id": "s_err", "message": "   "}).status_code == 422
    # 非法模型名触发 422
    assert client.post("/api/v1/chat", json={"session_id": "s_err", "message": "hi", "model": "unknown-llm"}).status_code == 422
    # 超界温度触发 422
    assert client.post("/api/v1/chat", json={"session_id": "s_err", "message": "hi", "temperature": 3.5}).status_code == 422

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：为 FastAPI 添加 API Key 请求头鉴权依赖（Depends）

**题目要求：**
使用 FastAPI `Header` 与 `Depends` 编写 `verify_api_key(x_api_key: str = Header(...))`，当请求头 `X-API-Key` 不等于 `'starmind-secret-2026'` 时抛出 `HTTPException(401)`，挂载到一个受保护路由 `/api/v1/secret` 并用 `TestClient` 验证 401 与 200。

**思路提示：** 在路由签名中写 `_: None = Depends(verify_api_key)`，测试时通过 `headers={'X-API-Key': ...}` 传参。


In [ ]:
from fastapi import Depends, FastAPI, Header, HTTPException
from fastapi.testclient import TestClient


def verify_api_key(x_api_key: str = Header(...)) -> str:
    if x_api_key != "starmind-secret-2026":
        raise HTTPException(status_code=401, detail="无效的 API Key")
    return x_api_key


app = FastAPI()


@app.get("/api/v1/secret")
def secret_endpoint(api_key: str = Depends(verify_api_key)):
    return {"status": "authorized", "key_prefix": api_key[:8]}


if __name__ == "__main__":
    c = TestClient(app)
    print("错误 Key:", c.get("/api/v1/secret", headers={"X-API-Key": "bad"}).status_code)
    print("正确 Key:", c.get("/api/v1/secret", headers={"X-API-Key": "starmind-secret-2026"}).json())


**答案解析与评分标准：**

`Depends` 依赖注入是 FastAPI 最优雅的特性之一，认证、数据库会话、限流器都可以写成可复用的依赖项。


### 作业 2（企业实战）：批量对话评测接口（Batch Chat Endpoint）

**题目要求：**
定义一个 Pydantic v2 模型 `BatchChatRequest(items: list[ChatRequest])`（限制单批最多 10 条），新增 `POST /api/v1/chat/batch` 端点批量处理并返回结果列表，用 `TestClient` 验证正常批处理和超过 10 条返回 422。

**思路提示：** 使用 `Field(..., min_length=1, max_length=10)` 限制列表长度。


In [ ]:
from typing import List
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
from src.day23_schemas import ChatRequest


class BatchChatRequest(BaseModel):
    items: List[ChatRequest] = Field(..., min_length=1, max_length=10)


app = FastAPI()


@app.post("/api/v1/chat/batch")
def batch_chat(payload: BatchChatRequest):
    return {
        "count": len(payload.items),
        "replies": [{"session_id": item.session_id, "echo": item.message} for item in payload.items],
    }


if __name__ == "__main__":
    c = TestClient(app)
    ok = c.post("/api/v1/chat/batch", json={"items": [{"session_id": "s1", "message": "hello"}]})
    print("批量结果:", ok.status_code, ok.json())


**答案解析与评分标准：**

给批处理接口强制加上 `max_length` 上限，是防止单个请求耗尽后端并发池的必要保护措施。


### 作业 3（架构进阶）：全局请求耗时与 Request-ID 追踪中间件

**题目要求：**
为 FastAPI 编写一个 HTTP 中间件 `@app.middleware('http')`，自动为每个响应注入 `X-Request-ID`（UUID）和 `X-Process-Time-Ms` 响应头，并用 `TestClient` 验证响应头存在。

**思路提示：** 在 `call_next(request)` 前后用 `time.perf_counter()` 计时，并将自定义头写入 `response.headers`。


In [ ]:
import time
import uuid
from fastapi import FastAPI, Request
from fastapi.testclient import TestClient

app = FastAPI()


@app.middleware("http")
async def add_observability_headers(request: Request, call_next):
    start = time.perf_counter()
    response = await call_next(request)
    elapsed_ms = round((time.perf_counter() - start) * 1000, 3)
    response.headers["X-Request-ID"] = uuid.uuid4().hex[:12]
    response.headers["X-Process-Time-Ms"] = str(elapsed_ms)
    return response


@app.get("/ping")
def ping():
    return {"pong": True}


if __name__ == "__main__":
    r = TestClient(app).get("/ping")
    print("Request-ID:", r.headers["X-Request-ID"], "| Time(ms):", r.headers["X-Process-Time-Ms"])


**答案解析与评分标准：**

生产排障时，前端把响应头里的 `X-Request-ID` 发给后端，就能在百万条日志中一秒定位那次报错请求。


---

## 🎯 老陈收尾复盘

今天大家把 FastAPI 路由、Pydantic v2 强校验和 `TestClient` 自动化测试全跑通了，是不是感觉比手写裸 socket 或老旧框架爽太多了？记住老陈的后端工程规范：所有入参必须经过 Pydantic 模型清洗和边界限制，所有错误状态码语义清晰（422 参数校验失败、404 会话不存在），所有接口上线前必须有 `TestClient` 用例护航！明天咱们上 SSE 流式推送和 SQLAlchemy 数据库持久化，完成第二阶段终极联调！
